# DBSCAN

O **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*), proposto por Martin Ester et al. em 1996, agrupa pontos por densidade local: procura regiões onde os pontos estão próximos entre si e as separa das regiões esparsas. Não recebe o número de clusters como entrada, encontra grupos de formato arbitrário e trata os pontos isolados como ruído.

## Fundamentação Matemática

O DBSCAN depende de dois parâmetros:

- $\varepsilon$ (`eps`): raio da vizinhança
- $\text{minPts}$: número mínimo de pontos para que uma vizinhança seja considerada densa

### Definições

**1. Vizinhança-$\varepsilon$**: para um ponto $p$, é o conjunto
$$N_\varepsilon(p) = \{q \in D \mid \text{dist}(p,q) \leq \varepsilon\}$$

**2. Ponto Central (Core Point)**: $p$ é um core point se
$$|N_\varepsilon(p)| \geq \text{minPts}$$

**3. Diretamente Alcançável por Densidade**: $q$ é diretamente alcançável a partir de $p$ se $q \in N_\varepsilon(p)$ e $p$ é um core point.

**4. Alcançável por Densidade**: $q$ é alcançável a partir de $p$ se existe uma cadeia $p_1, p_2, \dots, p_n$, com $p_1 = p$ e $p_n = q$, em que cada $p_{i+1}$ é diretamente alcançável a partir de $p_i$.

**5. Conectado por Densidade**: $p$ e $q$ são conectados por densidade se existe um ponto $o$ a partir do qual ambos são alcançáveis.

Um cluster é um conjunto maximal de pontos conectados por densidade. A relação de alcançabilidade não é simétrica: ela só se propaga *a partir* de core points. Daí os três tipos de ponto. Um **core point** satisfaz $|N_\varepsilon(p)| \geq \text{minPts}$, um **border point** não satisfaz mas está na vizinhança de algum core point, e um **noise point** não é nem um nem outro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN as SklearnDBSCAN, KMeans
from sklearn.datasets import fetch_california_housing, make_moons
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Implementação

Vamos implementar o algoritmo a partir das definições acima, usando apenas NumPy. A classe marca quais pontos são core e percorre os pontos em ordem: ao encontrar um core point ainda sem cluster, abre um novo e o expande por alcançabilidade, numa busca que só continua a partir de pontos que também são core.

In [ ]:
class DBSCAN:
    def __init__(self, eps=0.5, min_pts=5):
        self.eps = eps
        self.min_pts = min_pts

    def _distance_matrix(self, X):
        """Distância euclidiana entre todos os pares de pontos."""
        return np.linalg.norm(X[:, np.newaxis] - X, axis=2)

    def fit(self, X):
        D = self._distance_matrix(X)

        neighbors = [np.where(row <= self.eps)[0] for row in D]   # vizinhança-eps de cada ponto
        is_core = np.array([len(n) >= self.min_pts for n in neighbors])

        labels = np.full(len(X), -1)   # -1 = ruído
        cluster_id = 0

        for i in range(len(X)):
            if labels[i] != -1 or not is_core[i]:
                continue

            labels[i] = cluster_id     # core point ainda sem cluster: abre um novo
            queue = [i]

            while queue:               # expande por alcançabilidade a partir de i
                j = queue.pop()

                for k in neighbors[j]:
                    if labels[k] == -1:      # entra no primeiro cluster que o alcança
                        labels[k] = cluster_id

                        if is_core[k]:       # a busca só se propaga a partir de um core point
                            queue.append(k)

            cluster_id += 1

        self.labels_ = labels
        self.is_core_ = is_core
        self.n_clusters_ = cluster_id
        return self

### Dados Sintéticos

Um conjunto com três estruturas de formatos bem diferentes, um anel, dois blobs gaussianos e um arco, mais pontos espalhados uniformemente no papel de ruído. Nenhuma das formas é convexa.

Como vamos desenhar clusters várias vezes ao longo do notebook, isolamos também a plotagem numa função, que pinta o ruído de preto e circula os core points.

In [ ]:
rng = np.random.default_rng(42)


def arc(n, radius, a0, a1, spread, center=(0, 0)):
    """Pontos sobre um arco de raio médio `radius`, engrossado por ruído radial."""
    angle = rng.uniform(a0, a1, n)
    r = radius + rng.normal(0, spread, n)
    return np.c_[r * np.cos(angle), r * np.sin(angle)] + center


head = arc(400, 10, 0, 2*np.pi, 0.35)                                 # cabeça: o anel

eyes = np.vstack([rng.normal([-3.2, 3.0], 0.45, (50, 2)),             # olhos: dois blobs
                  rng.normal([3.2, 3.0], 0.45, (50, 2))])

mouth = arc(100, 5, np.deg2rad(200), np.deg2rad(340), 0.22, (0, -1))  # boca: arco inferior
mouth += rng.normal(0, [0.12, 0.15], mouth.shape)

noise = rng.uniform([-13, -13], [13, 13], (100, 2))                   # ruído uniforme

X_synthetic = np.vstack([head, eyes, mouth, noise])
true_labels = np.concatenate([np.full(400, 0), np.full(100, 1),
                              np.full(100, 2), np.full(100, -1)])

print(f"Shape: {X_synthetic.shape}")

In [ ]:
def plot_clusters(X, labels, ax, title, core_samples=None):
    """Desenha os clusters: ruído em preto e core points circulados."""
    palette = plt.cm.tab10.colors

    for label in np.unique(labels):
        mask = labels == label
        if label == -1:
            ax.scatter(X[mask, 0], X[mask, 1], c='black', marker='x',
                       s=30, alpha=0.6, label='Ruído')
        else:
            # a cor depende do rótulo, e não da posição, para não mudar entre gráficos
            ax.scatter(X[mask, 0], X[mask, 1], color=palette[label % len(palette)],
                       s=50, alpha=0.7, label=f'Cluster {label}')

    if core_samples is not None:
        ax.scatter(X[core_samples, 0], X[core_samples, 1], s=100,
                   facecolors='none', edgecolors='black', linewidth=1, alpha=0.5)

    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_clusters(X_synthetic, true_labels, axes[0], 'Estrutura Real')
axes[0].legend()

axes[1].scatter(X_synthetic[:, 0], X_synthetic[:, 1], c='black', alpha=0.6, s=30)
axes[1].set_title('Dados sem Rótulos')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

### Primeira Execução

In [ ]:
dbscan = DBSCAN(eps=1.3, min_pts=5).fit(X_synthetic)
labels = dbscan.labels_

fig, ax = plt.subplots(figsize=(7, 7))
plot_clusters(X_synthetic, labels, ax,
              f'DBSCAN (eps=1.3, min_pts=5): {dbscan.n_clusters_} clusters',
              core_samples=dbscan.is_core_)
ax.legend()
plt.show()

O algoritmo encontra **4 clusters** e descarta 53 pontos como ruído, sem receber o número de grupos. Ele separa os dois olhos, e com razão: são duas regiões densas desconectadas, e vê-las como uma coisa só é semântica nossa, não uma propriedade da densidade.

### Variando os Hiperparâmetros

Os dois parâmetros não têm o mesmo peso.

In [ ]:
eps_values = [0.5, 1.3, 2.0]
min_pts_values = [3, 5, 8]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i, eps in enumerate(eps_values):
    for j, min_pts in enumerate(min_pts_values):
        model = DBSCAN(eps=eps, min_pts=min_pts).fit(X_synthetic)
        plot_clusters(X_synthetic, model.labels_, axes[i, j],
                      f'eps={eps}, min_pts={min_pts}\n'
                      f'{model.n_clusters_} clusters, {np.sum(model.labels_ == -1)} ruído',
                      core_samples=model.is_core_)

plt.tight_layout()
plt.show()

O `eps` domina o resultado. Com `eps=0.5` a vizinhança é pequena demais e as estruturas se quebram em dezenas de fragmentos. Com `eps=2.0` os pontos de ruído entre a boca e o anel viram uma ponte, fundindo as duas num cluster de 571 pontos. É o **efeito corrente**: basta uma trilha de pontos ligando dois grupos para que virem um. A mesma propriedade que permite seguir formas alongadas torna o algoritmo frágil quando o ruído preenche o espaço entre os clusters.

O `min_pts` age como filtro, de forma bem mais suave: com `eps=1.3`, variá-lo de 3 a 8 quase não muda o resultado. Ele define quanta densidade se exige de um core point. A regra prática é começar com $\text{minPts} \geq D + 1$ para $D$ dimensões, ou $2D$, e depois ajustar o `eps`.

### Core, Border e Ruído

A distinção entre os três tipos determina o que se propaga e o que não se propaga na expansão dos clusters.

In [ ]:
core_mask = dbscan.is_core_
border_mask = (labels != -1) & ~core_mask
noise_mask = labels == -1

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(X_synthetic[core_mask, 0], X_synthetic[core_mask, 1],
                c='red', s=50, alpha=0.7, marker='o', label=f'Core ({core_mask.sum()})')
axes[0].scatter(X_synthetic[border_mask, 0], X_synthetic[border_mask, 1],
                c='blue', s=80, alpha=0.9, marker='s', label=f'Border ({border_mask.sum()})')
axes[0].scatter(X_synthetic[noise_mask, 0], X_synthetic[noise_mask, 1],
                c='black', s=40, alpha=0.7, marker='x', label=f'Ruído ({noise_mask.sum()})')
axes[0].set_title('Classificação dos Pontos')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()

plot_clusters(X_synthetic, true_labels, axes[1], 'Estrutura Real')

plt.tight_layout()
plt.show()

São 635 core points contra apenas 12 border points: as estruturas são espessas o suficiente para que quase todo ponto agrupado tenha vizinhança cheia. Os 53 pontos de ruído são, quase todos, os uniformes espalhados pelo quadro.

Há uma assimetria aqui. Core points e ruído são determinísticos, porque dependem só da contagem de vizinhos. Já um border point alcançável por dois clusters entra no que chegar primeiro, o que depende da ordem dos dados.

## Comparação com o K-Means

O DBSCAN não impõe forma alguma aos clusters, exige apenas conexão por densidade. Isso o separa dos métodos que atribuem cada ponto ao centróide mais próximo, onde as fronteiras de decisão são retas e os clusters, necessariamente convexos.

In [ ]:
X_moons, y_moons = make_moons(n_samples=400, noise=0.06, random_state=42)

kmeans_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)
dbscan_moons = DBSCAN(eps=0.2, min_pts=5).fit(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=40, alpha=0.8)
axes[0].set_title('Estrutura Real')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

plot_clusters(X_moons, kmeans_labels, axes[1], 'K-Means (K=2)')
plot_clusters(X_moons, dbscan_moons.labels_, axes[2],
              f'DBSCAN (eps=0.2, min_pts=5): {dbscan_moons.n_clusters_} clusters')

plt.tight_layout()
plt.show()

O K-Means corta as duas luas ao meio com uma reta, porque é a partição que minimiza a distância aos centróides, e o centro de massa de uma lua nem cai sobre ela. O DBSCAN recupera as duas exatamente e sem marcar nenhum ruído: cada lua é uma trilha contínua de pontos densos, e o vão entre elas é maior que o `eps`.

## O Gráfico K-Distance

Escolher o `eps` é a maior dificuldade prática do DBSCAN, e a heurística mais usada é o gráfico **K-Distance**. Para cada ponto, calcula-se a distância ao seu $k$-ésimo vizinho mais próximo, com $k = \text{minPts} - 1$, ordenam-se os valores e escolhe-se o `eps` na altura do cotovelo.

Pontos em regiões densas têm $d_k$ pequeno e formam a parte plana da curva. Pontos de borda e de ruído têm $d_k$ grande e formam a subida no final. O cotovelo é o limiar entre os dois regimes.

In [ ]:
def plot_k_distance(X, min_pts):
    """Distância de cada ponto ao seu (min_pts - 1)-ésimo vizinho, em ordem crescente."""
    nn = NearestNeighbors(n_neighbors=min_pts).fit(X)
    distances, _ = nn.kneighbors(X)          # a primeira coluna é o próprio ponto
    k_distances = np.sort(distances[:, -1])

    plt.figure(figsize=(10, 6))
    plt.plot(k_distances, linewidth=2)
    plt.xlabel('Pontos ordenados por distância')
    plt.ylabel(f'{min_pts - 1}-distance')
    plt.title(f'K-Distance Plot (min_pts={min_pts})')
    plt.show()


plot_k_distance(X_synthetic, min_pts=5)

A curva fica quase plana até cerca de 90% dos pontos e então dispara: a maioria tem seu 4º vizinho a menos de 1,1 de distância. O cotovelo cai perto de 1,3, exatamente o `eps` que usamos.

O método indica uma ordem de grandeza, não um valor exato. A região do cotovelo serve como intervalo de busca, e o valor final depende do que se quer chamar de ruído.

## Aplicação aos Distritos da Califórnia

O `fetch_california_housing` traz 20.640 distritos censitários com latitude e longitude. Usando só essas duas colunas, a pergunta vira geográfica: onde estão as aglomerações urbanas, sem informar quantas são nem onde procurar. Convertemos os graus em quilômetros, para que o `eps` tenha unidade física, e amostramos 3.000 distritos, já que nossa matriz de distâncias cresce com $N^2$.

In [ ]:
sample = fetch_california_housing(as_frame=True).frame.sample(3000, random_state=42)

lat = sample['Latitude'].values
lon = sample['Longitude'].values

# graus para quilômetros: um grau de longitude encolhe conforme a latitude
X_cal = np.c_[lon * 111 * np.cos(np.deg2rad(lat.mean())), lat * 111]

plot_k_distance(X_cal, min_pts=10)

A subida começa em torno de 10 km e a curva vira de vez perto de 20. Ficamos em 10 km, no início da rampa, que é a leitura mais restritiva do que conta como mancha urbana. Com 25 km a região da Baía já engole o corredor do vale central.

In [ ]:
dbscan_cal = DBSCAN(eps=10, min_pts=10).fit(X_cal)
kmeans_cal = KMeans(n_clusters=dbscan_cal.n_clusters_, n_init=10, random_state=42).fit_predict(X_cal)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

plot_clusters(X_cal, dbscan_cal.labels_, axes[0],
              f'DBSCAN (eps=10 km, min_pts=10): {dbscan_cal.n_clusters_} clusters')
plot_clusters(X_cal, kmeans_cal, axes[1], f'K-Means (K={dbscan_cal.n_clusters_})')

for ax in axes:
    ax.set_xlabel('Leste-Oeste (km)')
    ax.set_ylabel('Norte-Sul (km)')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
labels_cal = dbscan_cal.labels_
income = sample['MedInc'].values

for c in np.argsort(-np.bincount(labels_cal[labels_cal >= 0]))[:4]:
    mask = labels_cal == c
    print(f"cluster {c}: {mask.sum():5d} distritos, centro em ({lat[mask].mean():.2f}, {lon[mask].mean():.2f})")

print(f"\nRenda média: {income[labels_cal >= 0].mean():.2f} nos agrupados, "
      f"{income[labels_cal == -1].mean():.2f} nos {np.sum(labels_cal == -1)} de ruído")

São 19 clusters e 533 distritos como ruído, sem que o número de regiões tenha sido informado. Os quatro maiores são Los Angeles, a região da Baía, San Diego e Sacramento. O K-Means, com o mesmo K, precisa rotular todo ponto e gasta clusters fatiando o interior vazio do estado.

O ruído aqui não é erro de medição, é a Califórnia rural: renda média de 3.02, contra 4.03 nos distritos agrupados.

## Custo e Limitações

A matriz de distâncias tem $N^2$ entradas, cerca de 800 MB em `float64` para 10 mil pontos, e o pico durante o cálculo é várias vezes maior. Com índices espaciais, o Scikit-Learn consulta cada vizinhança em tempo logarítmico e roda em $O(N \log N)$, vantagem que se perde em dimensão alta, onde as árvores degeneram para a busca exaustiva.

A limitação central é trabalhar com uma densidade só. Os parâmetros fixam um único limiar para todo o conjunto, e se um grupo é denso e outro é esparso, nenhum par de valores atende aos dois: o `eps` que separa o denso transforma o esparso em ruído, e o que captura o esparso funde o denso com a vizinhança. É por isso que existem variantes como **OPTICS** e **HDBSCAN**, que trabalham com uma faixa de densidades.

Some-se a isso a alta sensibilidade ao `eps`, que é um raio no espaço das features, o que torna a padronização obrigatória quando as variáveis têm unidades diferentes, e o fato de que em dimensão alta todas as distâncias tendem a se parecer. Por fim, o determinismo é parcial: core points e ruído independem da ordem dos dados, a atribuição dos border points não.

## Exercícios

### Exercício 1: Ajuste de Parâmetros

Com as três esferas concêntricas geradas abaixo, construa o gráfico K-Distance para diferentes valores de `min_pts` e proponha um intervalo adequado para `eps`. Escolha os melhores valores, visualize os clusters em 3D e discuta como cada parâmetro influenciou a separação das estruturas.

A visualização usa o Plotly (`pip install plotly`), que permite girar o gráfico com o mouse, algo indispensável para enxergar cascas aninhadas.

In [ ]:
rng = np.random.default_rng(42)

radii = [3, 8, 12]
n_per_shell = 200
thickness = 0.4

shells = []

for r in radii:
    phi = rng.uniform(0, 2*np.pi, n_per_shell)                    # ângulo azimutal
    theta = np.arccos(rng.uniform(-1, 1, n_per_shell))            # ângulo polar
    rr = r + thickness * rng.standard_normal(n_per_shell)         # espessura da casca

    shells.append(np.c_[rr * np.sin(theta) * np.cos(phi),
                        rr * np.sin(theta) * np.sin(phi),
                        rr * np.cos(theta)])

X_spheres = StandardScaler().fit_transform(np.vstack(shells))
y_spheres = np.repeat(np.arange(len(radii)), n_per_shell)

print(f"Shape: {X_spheres.shape}")

In [ ]:
import plotly.express as px

fig = px.scatter_3d(x=X_spheres[:, 0], y=X_spheres[:, 1], z=X_spheres[:, 2])
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
# Seu código aqui

### Exercício 2: DBSCAN com Distância Radial

Com os mesmos dados, troque a distância euclidiana do `_distance_matrix` pela distância radial:

$$ d_{\text{radial}}(x_i, x_j) = \big| \, \|x_i\|_2 - \|x_j\|_2 \, \big| $$

Plote o K-Distance sob essa métrica para sugerir um `eps`, teste diferentes combinações de `eps` e `min_pts`, visualize os clusters em 3D e compare com o resultado euclidiano. Por que a métrica radial ajuda nesse tipo de estrutura, e o que ela deixaria de enxergar se as cascas não fossem concêntricas?

In [ ]:
# Seu código aqui

### Exercício 3: Detecção de Anomalias com DTW

O **DTW** (*Dynamic Time Warping*) mede a similaridade entre séries temporais alinhando os pontos de forma elástica, o que reconhece padrões semelhantes mesmo defasados ou em velocidades diferentes. A matriz de distâncias pode ser calculada com a `dtaidistance` (`pip install dtaidistance`):

```python
from dtaidistance import dtw

n = len(X)
D = np.zeros((n, n))

for i in range(n):
    for j in range(i + 1, n):
        D[i, j] = D[j, i] = dtw.distance_fast(X[i], X[j])
```

Aplique o DBSCAN com essa métrica ao conjunto de senóides abaixo, usando `SklearnDBSCAN(metric='precomputed')` com a matriz pronta ou adaptando o `_distance_matrix` da nossa classe. Ajuste `eps` e `min_pts` até separar as séries normais das anômalas e plote todas as séries destacando as detectadas como anomalia (`label = -1`).

In [ ]:
rng = np.random.default_rng(42)

n_series, n_outliers, length = 50, 2, 100
t = np.linspace(0, 4*np.pi, length)

series = []

for _ in range(n_series):                                          # senóides com amplitude,
    amp = rng.uniform(0.8, 1.2)                                    # frequência e fase variáveis
    freq = rng.uniform(0.9, 1.1)
    phase = rng.uniform(0, 0.5*np.pi)
    series.append(amp * np.sin(freq * t + phase) + 0.1 * rng.normal(size=length))

for _ in range(n_outliers):                                        # anomalias: amplitude e
    anomaly = rng.uniform(1.5, 2.0) * np.sin(rng.uniform(1.2, 1.5) * t)  # frequência fora da
    anomaly += 0.1 * rng.normal(size=length)                       # faixa, mais um pico
    anomaly[length // 2] += 3
    series.append(anomaly)

X_series = np.array(series)
y_series = np.array([0] * n_series + [-1] * n_outliers)

plt.figure(figsize=(10, 4))

for i in range(5):
    plt.plot(X_series[i], alpha=0.7, color='tab:blue', label='normal' if i == 0 else '')

for i in range(-n_outliers, 0):
    plt.plot(X_series[i], alpha=0.7, color='red', label='anomalia' if i == -1 else '')

plt.title('Séries Temporais com Anomalias')
plt.legend()
plt.show()

In [ ]:
# Seu código aqui